<a href="https://colab.research.google.com/github/erickcorrea3092/Antobiotics_Kurt_S_aureus/blob/main/2_Kurt_b%C3%BAsqueda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Antecedentes

## Imports

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import KFold, StratifiedShuffleSplit
from sklearn.svm import SVR
from sklearn.svm import NuSVR
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.feature_selection import (SelectFromModel, SelectKBest,
                                       VarianceThreshold)

## Globales

In [2]:
# 1 semilla para cosas random
RANDOM_STATE = 5712492

# 2 Minima varianza a conservar en las features
MIN_VARIANCE = 0

# 3 Separación
N_BINS=5 ##discretizer
N_SPLITS=10 ##splitter
TEST_SIZE=21/70 ##splitter

# 4 Algoritmo de escalamiento

# Otros que se pueden utilizar:StandardScaler(), MinMaxScaler() o Normalizer()
SCALER = StandardScaler()

# 5 Primer selector/transformador de features
feat_selector_model = RandomForestRegressor(n_estimators=49, min_samples_leaf=3, random_state=RANDOM_STATE)
FEAT_SELECTOR1 = SelectFromModel(feat_selector_model, max_features=22)

# 6 Segundo selector/transformador de features
n_pca = 8
FEAT_SELECTOR2 = PCA(n_components=n_pca, random_state=RANDOM_STATE)

# 7 Algoritmo de regresión
REGRESSOR = SVR(C=25, epsilon = 0.09)

# 8 Validación cruzada
METRICS = ["r2", "neg_mean_squared_error"]
CV_SPLITTER = KFold(n_splits=7, shuffle=True, random_state=RANDOM_STATE)



## Datos


In [3]:
# descargamos data
! gdown --id 1Lc6vEVZLOPGWuDQtl-LTQkBuaFNt5lWa
! gdown --id 1TP-26qH5Mn2RvzfMLHPGyMhNvuzkwzlW

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1Lc6vEVZLOPGWuDQtl-LTQkBuaFNt5lWa
To: /content/dataV.xlsx
100% 828k/828k [00:00<00:00, 22.1MB/s]
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1TP-26qH5Mn2RvzfMLHPGyMhNvuzkwzlW
To: /content/ActividadV.xlsx
100% 9.39k/9.39k [00:00<00:00, 20.1MB/s]


In [4]:
#Guardamos la data como una nueva variable
data = pd.read_excel("dataV.xlsx")
df_y = pd.read_excel("ActividadV.xlsx") #Esta es opcional opción 1
#df_y = data["Actividad"] #Esta es opcional opción 2

In [5]:
# Importamos datos
#_____________________________________________
df_x = data.copy()
#df_x = df_x.drop('Activity', axis=1) # Este renglón puede ser útil #Opción 2
df_x = df_x.select_dtypes("number")  # quitar non_numeric
#_____________________________________________
df_data = df_x.assign(Activity=df_y)
df_data

,nAcid,nBase,nAromAtom,nAromBond,nAtom,nHeavyAtom,nSpiro,nBridgehead,nHetero,nH,...,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,Activity
0,0,3,9,10,47,27,0,0,9,20,...,78.17227,405.1093,8.619347,2138,39,146,171,8.361111,5.750000,5.005513
1,0,0,12,12,26,18,0,0,5,8,...,65.48686,300.9851,11.576350,599,26,96,113,5.166667,3.916667,4.381642
2,0,0,15,16,36,23,0,0,6,13,...,73.78306,323.0728,8.974246,1245,35,126,151,6.138889,5.027778,4.114064
3,0,0,12,12,26,19,0,0,6,7,...,66.82214,318.9757,12.268290,693,28,102,120,6.027778,4.083333,5.012402
4,0,0,12,12,28,21,0,0,8,7,...,69.29945,345.9702,12.356080,929,32,112,132,7.138889,4.527778,5.047677
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,0,0,11,11,35,20,0,0,5,15,...,66.90812,350.0088,10.000250,915,26,98,111,7.166667,4.555556,4.243050
66,0,0,11,11,35,20,0,0,5,15,...,66.96854,306.0594,8.744553,902,27,98,112,7.166667,4.583333,4.087867
67,0,0,11,11,35,20,0,0,5,15,...,66.90812,290.0889,8.288255,915,26,98,111,7.166667,4.555556,4.161503
68,0,0,11,11,39,21,0,0,5,18,...,68.10269,302.1089,7.746382,1081,28,102,116,7.416667,4.888889,4.179135


## Load data

In [6]:
# Cargamos data
df_x = df_x.select_dtypes("number")  # quitar non_numeric
# Agregamos columna de actividad biológica
df_data = df_x.assign(activity=df_y)
# df_data

In [7]:
# dividimos train test con stratified
discretizer = sklearn.preprocessing.KBinsDiscretizer(n_bins=N_BINS, encode="ordinal", strategy="uniform")
splitter = StratifiedShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)
# Reshape y_discrete to be a 2D array (n_samples, 1)
y_discrete = discretizer.fit_transform(df_y.values.reshape(-1, 1))
split, split_test = next(splitter.split(df_x.values, y_discrete))
fractions = ["train" if i in split else "test" for i in range(df_data.shape[0])]
df_data = df_data.assign(fraction=fractions)
df_data

,nAcid,nBase,nAromAtom,nAromBond,nAtom,nHeavyAtom,nSpiro,nBridgehead,nHetero,nH,...,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,activity,fraction
0,0,3,9,10,47,27,0,0,9,20,...,405.1093,8.619347,2138,39,146,171,8.361111,5.750000,5.005513,train
1,0,0,12,12,26,18,0,0,5,8,...,300.9851,11.576350,599,26,96,113,5.166667,3.916667,4.381642,test
2,0,0,15,16,36,23,0,0,6,13,...,323.0728,8.974246,1245,35,126,151,6.138889,5.027778,4.114064,train
3,0,0,12,12,26,19,0,0,6,7,...,318.9757,12.268290,693,28,102,120,6.027778,4.083333,5.012402,test
4,0,0,12,12,28,21,0,0,8,7,...,345.9702,12.356080,929,32,112,132,7.138889,4.527778,5.047677,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,0,0,11,11,35,20,0,0,5,15,...,350.0088,10.000250,915,26,98,111,7.166667,4.555556,4.243050,train
66,0,0,11,11,35,20,0,0,5,15,...,306.0594,8.744553,902,27,98,112,7.166667,4.583333,4.087867,train
67,0,0,11,11,35,20,0,0,5,15,...,290.0889,8.288255,915,26,98,111,7.166667,4.555556,4.161503,train
68,0,0,11,11,39,21,0,0,5,18,...,302.1089,7.746382,1081,28,102,116,7.416667,4.888889,4.179135,train


In [8]:
# sacamos los vectores (x, y) de interés
x_train = df_x[df_data["fraction"] == "train"].values
y_train = df_y[df_data["fraction"] == "train"].values
x_test = df_x[df_data["fraction"] == "test"].values
y_test = df_y[df_data["fraction"] == "test"].values
y_train = np.reshape(y_train, -1)
y_test = np.reshape(y_test, -1)

x_train = x_train.astype(float)
y_train = y_train.astype(float)
x_test = x_test.astype(float)
y_test = y_test.astype(float)

# x_train





# Train & Evaluate

In [9]:
# creamos pipeline: el modelo final final
steps = [
    ("var0", VarianceThreshold(MIN_VARIANCE)),
    ("scaling", SCALER),
    ("feat_selector1", FEAT_SELECTOR1),
    ("feat_selector2", FEAT_SELECTOR2),
    ("regressor", REGRESSOR)
]
pipeline = Pipeline(steps, verbose=True)
pipeline = pipeline.fit(x_train, y_train)
pipeline.set_params(verbose=False)
pipeline

[Pipeline] .............. (step 1 of 5) Processing var0, total=   0.0s
[Pipeline] ........... (step 2 of 5) Processing scaling, total=   0.0s
[Pipeline] .... (step 3 of 5) Processing feat_selector1, total=   0.6s
[Pipeline] .... (step 4 of 5) Processing feat_selector2, total=   0.0s
[Pipeline] ......... (step 5 of 5) Processing regressor, total=   0.0s


Pipeline(steps=[('var0', VarianceThreshold(threshold=0)),
                ('scaling', StandardScaler()),
                ('feat_selector1',
                 SelectFromModel(estimator=RandomForestRegressor(min_samples_leaf=3,
                                                                 n_estimators=49,
                                                                 random_state=5712492),
                                 max_features=22)),
                ('feat_selector2', PCA(n_components=8, random_state=5712492)),
                ('regressor', SVR(C=25, epsilon=0.09))])

In [10]:
# analizamos paso a paso las transformaciones
z = x_train
for (i, step) in enumerate(pipeline[:-1]):
    print(f"Step {pipeline.steps[i][0]} ({i}): {step.__class__.__name__}")
    print(f"  Input: {z.shape}. Output: {step.transform(z).shape}")
    z = step.transform(z)

Step var0 (0): VarianceThreshold
  Input: (49, 1218). Output: (49, 1010)
Step scaling (1): StandardScaler
  Input: (49, 1010). Output: (49, 1010)
Step feat_selector1 (2): SelectFromModel
  Input: (49, 1010). Output: (49, 22)
Step feat_selector2 (3): PCA
  Input: (49, 22). Output: (49, 8)


# resultados


In [11]:
# validar accuracy
metric = "r2"
# cv = sklearn.model_selection.LeaveOneOut()
cv = sklearn.model_selection.KFold(n_splits=7, shuffle=True, random_state=RANDOM_STATE)
cv_scores = sklearn.model_selection.cross_val_score(pipeline, x_train, y_train, cv=cv, verbose=2, scoring=metric)
scorer = sklearn.metrics.get_scorer(metric)
pd.DataFrame([{"train": scorer(pipeline, x_train, y_train),
               "cv": np.mean(cv_scores),
               "test": scorer(pipeline, x_test, y_test),
               "metric": metric}])

[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.4s


[Parallel(n_jobs=1)]: Done   7 out of   7 | elapsed:    3.3s finished


,train,cv,test,metric
0,0.988278,0.67806,0.748522,r2
